# Scale and Automate Config Generation (Notebook 3)

This notebook generates Kaiaulu config files for each main project repo in the GHTorrent database.

**What this notebook does:**
1. Queries MySQL/GHTorrent to identify canonical repos with sentiment-labeled comments
2. Generates a `.yml` config file per repo (using `trinitycore.yml` as a template) and writes them to Kaiaulu's `conf/` directory

**What comes next** — once configs are written, use these Kaiaulu vignettes to download and parse comments:
- `vignettes/download_github_events.Rmd` → commit comments
- `vignettes/download_github_pull_request_comments.Rmd` → PR inline comments

### Planned Output

1. One `.yml` config file per main project repo in the GHTorrent database, written to Kaiaulu's `conf/` directory.

### Step 1: Import Dependencies

In [ ]:
import os
import subprocess
from pathlib import Path

import pandas as pd
import yaml
from sqlalchemy import create_engine, text

### Step 2: Set Paths and Configuration

Update the variables below before running:
- **`KAIAULU_REPO`** — path to your local Kaiaulu repo
- **`MYSQL_DB`** / **`MYSQL_PASSWORD`** — your database credentials
- **`MAX_REPOS`** — set to an integer to limit the number of repos processed, or `None` to process all
- **`WRITE_CONFIGS`** — set to `False` to do a dry run without writing any files

In [ ]:
# Paths
KAIAULU_REPO = (Path(".").resolve() / ".." / "kaiaulu").resolve()

# Kaiaulu-owned inputs/outputs
CONF_DIR = KAIAULU_REPO / "conf"
TEMPLATE_PATH = CONF_DIR / "trinitycore.yml"

# Repo selection cap (None = all main project repos)
MAX_REPOS = None

# MySQL connection (override with env vars if needed)
MYSQL_HOST = os.getenv("MYSQL_HOST", "localhost")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_DB = os.getenv("MYSQL_DB", "ADD_DB_NAME_HERE")
MYSQL_USER = os.getenv("MYSQL_USER", "root")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD", "ADD_PASSWORD_HERE")

# Toggle writing config files to Kaiaulu conf/
WRITE_CONFIGS = True

### Step 3: Query Canonical Repos from GHTorrent

Queries MySQL to find main (non-fork) repos that have at least one sentiment-labeled comment (commit or PR). Results are loaded into `repos`.

Expected output (~82 repos):

| | owner | repo |
|---|---|---|
| 0 | akka | akka |
| 1 | antirez | redis |
| 2 | ariya | phantomjs |
| 3 | automapper | automapper |
| 4 | bartaz | impress.js |

In [ ]:
# Query canonical repos that have sentiment-labeled comments
engine = create_engine(
    f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DB}'
 )

sql = """
WITH RECURSIVE project_root AS (
    SELECT p.id AS project_id, p.id AS root_id
    FROM projects p
    WHERE p.forked_from IS NULL
    UNION ALL
    SELECT c.id AS project_id, pr.root_id
    FROM projects c
    JOIN project_root pr ON c.forked_from = pr.project_id
),
comment_project_rows AS (
    SELECT cs.ID AS comment_id, c.project_id
    FROM comment_sentiment cs
    JOIN commit_comments cc ON cs.ID = cc.comment_id
    JOIN commits c ON cc.commit_id = c.id
    UNION ALL
    SELECT cs.ID AS comment_id, pr.base_repo_id AS project_id
    FROM comment_sentiment cs
    JOIN pull_request_comments prc ON cs.ID = prc.comment_id
    JOIN pull_requests pr ON prc.pull_request_id = pr.id
    UNION ALL
    SELECT cs.ID AS comment_id, pr.head_repo_id AS project_id
    FROM comment_sentiment cs
    JOIN pull_request_comments prc ON cs.ID = prc.comment_id
    JOIN pull_requests pr ON prc.pull_request_id = pr.id
)
SELECT DISTINCT LOWER(u.login) AS owner, LOWER(p.name) AS repo
FROM comment_project_rows cpr
JOIN project_root pr ON pr.project_id = cpr.project_id
JOIN projects p ON p.id = pr.root_id
JOIN users u ON u.id = p.owner_id
ORDER BY owner, repo
"""

repos = pd.read_sql(text(sql), con=engine)
print('repos found:', len(repos))
repos.head()

canonical repos found: 82


,owner,repo
0,akka,akka
1,antirez,redis
2,ariya,phantomjs
3,automapper,automapper
4,bartaz,impress.js


### Step 4: Generate and Write Config Files

Builds a `.yml` config file for each repo using `trinitycore.yml` as a template and writes it to Kaiaulu's `conf/` directory.

Each config follows this structure:
```yaml
project:
  website: https://github.com/{owner}/{repo}
issue_tracker:
  github:
    project_key_1:
      owner: {owner}
      repo: {repo}
      issue_or_pr_comment: rawdata/github/{owner}/{repo}/issue_or_pr_comment/
      issue_event: rawdata/github/{owner}/{repo}/issue_event/
      commit: rawdata/github/{owner}/{repo}/commit/
      commit_comments: rawdata/github/{owner}/{repo}/commit_comments/
      pr_comments: rawdata/github/{owner}/{repo}/pr_comments/
```

Expected output: a list of written `.yml` filenames, e.g. `['akka.yml', 'redis.yml', ...]`

In [ ]:
# Build YAML configs for 82 project repos using trinitycore.yml as the base template
header_lines = [
    "# -*- yaml -*-",
    "# https://github.com/sailuh/kaiaulu",
    "#",
    "# Copying and distribution of this file, with or without modification,",
    "# are permitted in any medium without royalty provided the copyright",
    "# notice and this notice are preserved.  This file is offered as-is,",
    "# without any warranty.",
    "",
    "# Project Configuration File #",
    "#",
    "# To perform analysis on open source projects, you need to manually",
    "# collect some information from the project's website. As there is",
    "# no standardized website format, this file serves to distill",
    "# important data source information so it can be reused by others",
    "# and understood by Kaiaulu.",
    "#",
    "# Please check https://github.com/sailuh/kaiaulu/tree/master/conf to",
    "# see if a project configuration file already exists. Otherwise, we",
    "# would appreciate if you share your curated file with us by sending a",
    "# Pull Request: https://github.com/sailuh/kaiaulu/pulls",
    "#",
    "# Note, you do NOT need to specify this entire file to conduct analysis.",
    "# Each R Notebook uses a different portion of this file. To know what",
    "# information is used, see the project configuration file section at",
    "# the start of each R Notebook.",
    "#",
    "# Please comment unused parameters instead of deleting them for clarity.",
    "# If you have questions, please open a discussion:",
    "# https://github.com/sailuh/kaiaulu/discussions",
    "",
]

def build_conf(template, owner, repo):
    conf = template.copy()
    conf.setdefault("project", {})
    conf["project"]["website"] = f"https://github.com/{owner}/{repo}"

    conf.setdefault("issue_tracker", {})
    conf["issue_tracker"].setdefault("github", {})
    conf["issue_tracker"]["github"].setdefault("project_key_1", {})
    conf["issue_tracker"]["github"]["project_key_1"]["owner"] = owner
    conf["issue_tracker"]["github"]["project_key_1"]["repo"] = repo

    # Keep relative paths so data lands under backend cwd (sentiment_github_dataset)
    base_path = f"rawdata/github/{owner}/{repo}"
    conf["issue_tracker"]["github"]["project_key_1"]["issue_or_pr_comment"] = f"{base_path}/issue_or_pr_comment/"
    conf["issue_tracker"]["github"]["project_key_1"]["issue_event"] = f"{base_path}/issue_event/"
    conf["issue_tracker"]["github"]["project_key_1"]["commit"] = f"{base_path}/commit/"
    conf["issue_tracker"]["github"]["project_key_1"]["commit_comments"] = f"{base_path}/commit_comments/"
    conf["issue_tracker"]["github"]["project_key_1"]["pr_comments"] = f"{base_path}/pr_comments/"
    return conf

with open(TEMPLATE_PATH, "r", encoding="utf-8") as f:
    template_conf = yaml.safe_load(f)

if MAX_REPOS is None:
    pilot = repos.copy()
else:
    pilot = repos.head(MAX_REPOS)

print(f"repos selected for config generation: {len(pilot)}")

written = []
for row in pilot.itertuples(index=False):
    owner = row.owner
    repo = row.repo
    target_path = CONF_DIR / f"{repo}.yml"
    conf = build_conf(template_conf, owner, repo)
    yaml_body = yaml.safe_dump(conf, sort_keys=False)
    if WRITE_CONFIGS:
        with open(target_path, "w", encoding="utf-8") as out:
            out.write("\n".join(header_lines))
            out.write("\n")
            out.write(yaml_body)
        written.append(target_path.name)
print("written configs:", written)

repos selected for config generation: 82
written configs: ['akka.yml', 'redis.yml', 'phantomjs.yml', 'automapper.yml', 'impress.js.yml', 'bitcoin.yml', 'boto.yml', 'craftbukkit.yml', 'cakephp.yml', 'compass.yml', 'clojure.yml', 'slim.yml', 'diaspora.yml', 'django-cms.yml', 'django.yml', 'django-debug-toolbar.yml', 'elasticsearch.yml', 'codeigniter.yml', 'facebook-android-sdk.yml', 'folly.yml', 'hiphop-php.yml', 'php-sdk.yml', 'tornado.yml', 'thinkup.yml', 'android.yml', 'gitlabhq.yml', 'html5-boilerplate.yml', 'devtools.yml', 'chosen.yml', 'sparkleshare.yml', 'octopress.yml', 'actionbarsherlock.yml', 'blueprint-css.yml', 'http-parser.yml', 'libuv.yml', 'node.yml', 'jquery.yml', 'requests.yml', 'beanstalkd.yml', 'libgit2.yml', 'ccv.yml', 'mangos.yml', 'd3.yml', 'memcached.yml', 'sick-beard.yml', 'flask.yml', 'jekyll.yml', 'mongo.yml', 'mono.yml', 'plupload.yml', 'three.js.yml', 'homebrew.yml', 'nancy.yml', 'storm.yml', 'netty.yml', 'openframeworks.yml', 'devise.yml', 'rails.yml', 'reddi

### When to Move On to Notebook 4

Move to Notebook 4 after all of the following are true:

1. The 82 `.yml` files generated from Step 4 exist in Kaiaulu's `conf/` directory.
4. Spot-check a few configs to confirm the `owner`, `repo`, and `rawdata/` paths are populated correctly and follow the formatting indicated in Step 4.